# Simple and Multiple Linear Regression

This notebook creates a reproducible synthetic dataset and compares two scikit-learn regression models. It generates `regression_data.csv` in the notebook's working folder, loads that file into pandas, trains both models using training data only, and evaluates their predictions on held-out test data.

## 1. Import the tools we need

NumPy generates the sample data, pandas organizes it, scikit-learn splits and models it, and Matplotlib creates the comparison plots.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

## 2. Generate and save a reproducible dataset

There are 5,000 observations, five numerical input features, and one continuous target (`target`). The target is a weighted sum of all five features plus random noise. Because the same fixed random seed is used each time, rerunning this cell creates the same data.

In [ ]:
random_generator = np.random.default_rng(seed=42)
number_of_observations = 5_000

features = random_generator.normal(
    loc=0.0,
    scale=1.0,
    size=(number_of_observations, 5),
)
noise = random_generator.normal(loc=0.0, scale=2.5, size=number_of_observations)
target = (
    8.0
    + 2.4 * features[:, 0]
    - 2.0 * features[:, 1]
    + 1.25 * features[:, 2]
    + 0.75 * features[:, 3]
    - 1.5 * features[:, 4]
    + noise
)

column_names = [f"feature_{number}" for number in range(1, 6)]
generated_data = pd.DataFrame(features, columns=column_names)
generated_data["target"] = target

csv_path = Path.cwd() / "regression_data.csv"
generated_data.to_csv(csv_path, index=False)
print(f"Saved {len(generated_data):,} observations to {csv_path}")

## 3. Load the CSV and inspect the data

Reading the saved file back in demonstrates the common workflow of loading a dataset from disk before analysis. The shape is shown as `(rows, columns)`.

In [ ]:
data = pd.read_csv(csv_path)
print(f"Data shape (rows, columns): {data.shape}")
data.head()

## 4. Simple vs. multiple linear regression

**Simple Linear Regression (SLR)** uses one input feature to predict a target. Here, it uses only `feature_1`. **Multiple Linear Regression (MLR)** uses two or more input features; here it uses all five features. Since the synthetic target depends on all five, MLR has access to more of the information used to create it, while SLR provides a useful one-feature baseline.

## 5. Split into training and test sets

The split is performed once on all features. Both models use the same training and test observations, making their results easier to compare. The test data is held back from fitting and is used only for evaluation.

In [ ]:
X = data[column_names]
y = data["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print(f"Training observations: {len(X_train):,}")
print(f"Test observations: {len(X_test):,}")

## 6. Train both models and predict the test set

The SLR model receives only `feature_1`; the MLR model receives all five features. Each model is fitted only with its training rows, then predicts the shared held-out test rows.

In [ ]:
slr_model = LinearRegression()
slr_model.fit(X_train[["feature_1"]], y_train)
slr_predictions = slr_model.predict(X_test[["feature_1"]])

mlr_model = LinearRegression()
mlr_model.fit(X_train, y_train)
mlr_predictions = mlr_model.predict(X_test)

print(f"SLR coefficient for feature_1: {slr_model.coef_[0]:.3f}")
print(f"SLR intercept: {slr_model.intercept_:.3f}")
print("MLR coefficients:", dict(zip(column_names, mlr_model.coef_.round(3))))
print(f"MLR intercept: {mlr_model.intercept_:.3f}")

## 7. Evaluate the test-set predictions

- **Mean Squared Error (MSE):** the average of squared prediction errors. Lower is better; squaring gives larger errors more influence, and the units are squared target units.
- **Mean Absolute Error (MAE):** the average absolute prediction error. Lower is better, and it is in the same units as the target, so it is often an intuitive typical-error measure.
- **R² score:** the proportion of target variation explained by the model compared with predicting the test-set mean. A score near 1 is strong, 0 is about as good as the mean baseline, and a negative score is worse than that baseline.

These metrics are calculated from test data, not the training data, to estimate how each model performs on observations it did not fit.

In [ ]:
model_predictions = {
    "Simple Linear Regression": slr_predictions,
    "Multiple Linear Regression": mlr_predictions,
}

metrics = pd.DataFrame(
    {
        model_name: {
            "MSE": mean_squared_error(y_test, predictions),
            "MAE": mean_absolute_error(y_test, predictions),
            "R² score": r2_score(y_test, predictions),
        }
        for model_name, predictions in model_predictions.items()
    }
).T

metrics.round(3)

## 8. Compare actual and predicted values for 10 test observations

Each plot shows exactly 10 held-out test observations. A vertical connector joins the predicted value to the actual value, and its annotation gives the residual calculated as `actual - predicted`. A residual above zero means the model predicted too low; one below zero means it predicted too high. Residuals close to zero indicate more accurate predictions, while larger absolute residuals indicate larger errors.

In [ ]:
def plot_test_observations(model_name, predictions):
    """Plot actual and predicted values and residuals for 10 test rows."""
    actual_values = y_test.iloc[:10].to_numpy()
    predicted_values = predictions[:10]
    observation_numbers = np.arange(1, 11)
    residuals = actual_values - predicted_values

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(
        observation_numbers,
        actual_values,
        marker="o",
        linewidth=2,
        label="Actual target",
    )
    ax.plot(
        observation_numbers,
        predicted_values,
        marker="s",
        linewidth=2,
        label="Predicted target",
    )
    ax.vlines(
        observation_numbers,
        predicted_values,
        actual_values,
        color="gray",
        alpha=0.7,
        label="Residual (actual - predicted)",
    )

    for x_position, actual, predicted, residual in zip(
        observation_numbers, actual_values, predicted_values, residuals
    ):
        midpoint = (actual + predicted) / 2
        ax.annotate(
            f"{residual:+.2f}",
            (x_position, midpoint),
            xytext=(5, 0),
            textcoords="offset points",
            fontsize=8,
            va="center",
        )

    ax.set_title(f"{model_name}: Actual vs. Predicted (10 Test Observations)")
    ax.set_xlabel("Test observation number")
    ax.set_ylabel("Target value")
    ax.set_xticks(observation_numbers)
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()


plot_test_observations("Simple Linear Regression", slr_predictions)
plot_test_observations("Multiple Linear Regression", mlr_predictions)

## 9. Interpreting the results

Look for actual and predicted markers that are close together and residual connectors that are short. A **positive residual** (`actual - predicted > 0`) means the actual value is greater than the prediction, so the model underpredicted. A **negative residual** means the prediction is greater than the actual value, so the model overpredicted. The residual's absolute size is the size of that observation's prediction error.

Because all five features contribute to the generated target, MLR will generally have lower test MSE and MAE and a higher R² than SLR. The random noise means neither model is expected to predict every observation exactly. Compare the metric table and the two plots together: the table summarizes all test observations, while each plot illustrates the errors for the first 10 test observations.